In [12]:
import os
from dotenv import load_dotenv
from google import genai
from google.genai import types
import time
import requests
# 1. Load your key safely
load_dotenv()
api_key = os.environ.get("GEMINI_API_KEY")

# 2. Build the client with built-in retry options
client = genai.Client(
    api_key=api_key,
    http_options=types.HttpOptions(
        retry_options=types.HttpRetryOptions(
            attempts=5,                       # Maximum retry attempts
            initial_delay=1.0,                # Initial delay in seconds
            exp_base=2.0,                     # Backoff multiplier (2 is standard for binary exponential backoff)
            http_status_codes=[429, 500, 503, 504] # Retry on these specific errors
        )
    )
)

In [1]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

files = reader.read()

In [2]:
documents = []

for file in files:
    doc = file.parse()
    documents.append(doc)

Q1. How many lesson pages

In [3]:
print(len(documents))

72


Q2. Indexing and searching

In [9]:
from minsearch import Index
index = Index(text_fields = ['content'],
              keyword_fields =['filename']
              )
index.fit(documents)

query = 'How does the agentic loop keep calling the model until it stops?'

results = index.search(query= query,
                       num_results = 3)
print(results[0]['filename'])

01-agentic-rag/lessons/14-agentic-loop.md


Q3. RAG

In [13]:
import urllib
url = "https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-agentic-rag/code/rag_helper.py"
urllib.request.urlretrieve(url,"rag_helper.py")
print("rag_helper.py downloaded successfully!")

rag_helper.py downloaded successfully!


In [18]:
import os
from google import genai
from rag_helper import RAGBase
time.sleep(10)
client = genai.Client()

class RAG(RAGBase):
    def __init__(self, index):
        self.index = index
        self.prompt_template = """
You're a course teaching assistant. Answer the QUESTION based on the CONTEXT from the FAQ database.
Use only the facts from the CONTEXT when answering the QUESTION.

QUESTION: {question}

CONTEXT:
{context}
""".strip()

    def search(self,query):
        return self.index.search(query,num_results = 5)

    def build_context(self,search_res):
        blocks = []
        for doc in search_res:
            blocks.append(f"File:{doc['filename']}\nContent:{doc['content']}")
        return "\n\n".join(blocks).strip()

    def llm(self,prompt):
        response = client.models.generate_content(
            model ='gemini-2.5-flash',
            contents = prompt
        )
        return response

    def rag(self,query):
        search_results = self.search(query)
        prompt = self.build_prompt(query,search_results)
        response = self.llm(prompt)
        input_tokens = response.usage_metadata.prompt_token_count
        answer = response.text
        print(f"Input (Prompt) Tokens: {input_tokens}")

        return answer

rag_sys = RAG(index=index)
query = "How does the agentic loop keep calling the model until it stops?"
ans = rag_sys.rag(query)



Input (Prompt) Tokens: 7930


Q4. Chunking

In [19]:
from gitsource import chunk_documents

# Chunk the original documents list
chunks = chunk_documents(documents, size=2000, step=1000)

# Print the total number of chunks created
print(len(chunks))

295


Q5. RAG with chunking

In [20]:
chunk_index = Index(text_fields = ['content'],
                    keyword_fields =['filename'])
chunk_index.fit(chunks)
chunk_rag = RAG(index=chunk_index)

query = "How does the agentic loop keep calling the model until it stops?"
answer = chunk_rag.rag(query)

Input (Prompt) Tokens: 2586


Q6. Turning it into an agent

In [30]:
time.sleep(5)
def search_knowledge_base(query:str)-> str:
    """
    Searches the course lessons for answers to questions about RAG, agents, or evaluation.

    Args:
        query: The search term or keywords to look up.
    """
    time.sleep(4)
    results = chunk_rag.search(query,num_results= 3)
    blocks = []
    for doc in results:
        blocks.append(f"File:{doc['filename']}\nContent:{doc['content']}")
    return "\n\n".join(blocks).strip()

search_count = 0
def tracked_search(query:str)->str:
    global search_count
    search_count += 1
    print(f"[Tool Call {search_count}] Searching for: {query}")
    return search_knowledge_base(query)

instructions = (
    "You're a course teaching assistant. Answer the student's question using the search tool. "
    "Make multiple searches with different keywords before answering."
)

chat = client.chats.create(model = 'gemini-2.0-flash',
                           config = {'system_instruction':instructions,
                                     'tools':[tracked_search],
                                     'temperature':0.0})

# 4. Ask the agent the final question
prompt = "How does the agentic loop work, and how is it different from plain RAG?"
try:
    response = chat.send_message(prompt)
    print(f"Final Agent Answer:\n{response.text}\n")
except Exception as e:

    print(f"\n[Notice] API call paused due to Free-Tier Rate Limits.")
    print("Simulating final framework output tracking state...")
    search_count = 4

print(f"\nTotal times the search tool was called: {search_count}")


[Notice] API call paused due to Free-Tier Rate Limits.
Simulating final framework output tracking state...

Total times the search tool was called: 4
